# Chốt ngân sách token cho SFT — UIT DSC 2026 Task 2 v10

**Đây là lần chạy ĐO, không phải train.** Câu hỏi duy nhất cần trả lời:
`max_seq_len` bao nhiêu, và `top_n`/`max_chars` nào cho tỉ lệ gold cao nhất mà vẫn vừa.

Máy dev không có GPU nên chỉ đo được BM25 thuần, cho gold-hit 74,6% ở cấu hình
`top_n=5 max_chars=12.000`. §15.3 đã đo cross-encoder cho **+18,8 R@1 / +11,0 R@5**
trên cùng pool, nên con số đó là **cận dưới** — và thứ tự ưu tiên giữa các cấu hình
có thể đổi. Notebook này đo lại có rerank.

Cần gắn trước khi chạy:

| loại | thứ cần gắn | dùng làm gì |
|---|---|---|
| Accelerator | **GPU T4 x2** | XLM-R large 568M |
| Internet | **ON** | tải reranker + tokenizer Qwen |
| Input | dữ liệu thi (`selected-contexts/` + `train.json`) | dựng corpus/chunk/gold |
| Input | dataset `legalqa-v10-src` | codebase |
| Model | `qwen3-5-2b-base` | tokenizer.json để đếm token THẬT |


## 1 — Nạp codebase từ Dataset

Codebase vào bằng **Kaggle Dataset** `legalqa-v10-src`, không clone GitHub: repo private,
fine-grained PAT qua Kaggle Secrets tốn hai vòng debug mà vẫn 404.

Cần gắn: **Input → Add Input → Datasets → `legalqa-v10-src`**

Dòng `codebase:` in ra commit của bản đang chạy. Đối chiếu với `git log -1` trên máy dev —
lệch nghĩa là dataset cũ, chạy `./scripts/push_dataset.sh` rồi *Add Input* lại bản mới.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Không clone GitHub nữa: repo private, fine-grained PAT + Kaggle Secrets tốn hai vòng
# debug mà vẫn 404. Codebase đi vào bằng Kaggle Dataset — không auth, chạy được cả khi
# Internet OFF. Đẩy bản mới bằng scripts/push_dataset.sh trên máy dev.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    # Kaggle đôi khi giữ nguyên .zip thay vì bung; tự bung vào working.
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

# Copy sang working: scripts ghi bm25.pkl/chunks.jsonl, /kaggle/input là read-only.
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")
print("số GPU  :", torch.cuda.device_count())

# P100 (sm_60) không chạy được fp16 với torch bản trên Kaggle. Code tự tụt về
# fp32 nên không vỡ, nhưng chậm ~2x — báo to để biết mà đổi Accelerator.
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy -> fp32, đổi Accelerator sang T4 x2)")


## 2 — Tìm dữ liệu thi

Dò thay vì bắt gõ tay đường dẫn: tên dataset khác nhau tuỳ bạn gắn competition hay upload zip.
Mốc nhận dạng là thư mục `selected-contexts/` — thứ duy nhất chắc chắn có.

In [ ]:
from pathlib import Path

cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

train = DATA_DIR / "train.json"
assert train.exists(), f"Thiếu train.json cạnh selected-contexts/ trong {DATA_DIR}"

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["TRAIN_JSON"] = str(train)
os.environ["CLEAN_DIR"] = "/kaggle/working/clean"
os.environ["PYTHONPATH"] = "src"

n = len(list((DATA_DIR / "selected-contexts").glob("*.json")))
print(f"DATA_DIR  {DATA_DIR}")
print(f"corpus    {n:,} file  (máy dev: 8.532)")

## 3 — clean → chunk → gold (CPU, ~12 phút)

Ba số phải khớp máy dev, lệch là dữ liệu khác bản — mọi số sau đó vô nghĩa:
`Document 8.512` · `Chunk 287.358` · `gold 6.250/7.000 = 89,3%`

In [ ]:
# Dùng lại clean/ của lần chạy trước nếu có (gắn kernel_sources): dựng lại mất 26 phút,
# và kernel bật GPU thì 26 phút đó đốt quota GPU dù không chạm GPU. Ngân sách 30 GPU-h/tuần.
from pathlib import Path
cu = next((p for p in Path("/kaggle/input").glob("**/clean") if (p / "chunks.jsonl").exists()), None)
if cu:
    os.environ["CLEAN_DIR"] = str(cu)
    print("dùng lại clean/ của lần chạy trước:", cu)
    print("  ", sorted(p.name for p in cu.iterdir()))
else:
    !mkdir -p $CLEAN_DIR
    !python scripts/clean_data.py
    !python scripts/build_chunks.py
    !python scripts/build_gold.py


## 4 — Lấy reranker

`AITeamVN/Vietnamese_Reranker` — XLM-R large, 567.755.777 params, `XLMRobertaForSequenceClassification`,
head hồi quy có sẵn nên **không train gì cả**. Với Qwen 3B là 3,66B, còn trống 340M dưới trần 4B.

Ưu tiên bản đã gắn sẵn trong `/kaggle/input` (chạy được cả khi Internet OFF); không có thì tải HF.

In [ ]:
import json

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    MODEL = str(local[0])
    print("dùng bản đã gắn:", MODEL)
else:
    from huggingface_hub import snapshot_download
    MODEL = snapshot_download("AITeamVN/Vietnamese_Reranker")
    print("tải từ HuggingFace:", MODEL)

cfg = json.loads((Path(MODEL) / "config.json").read_text())
print(cfg["architectures"], "layers", cfg["num_hidden_layers"], "hidden", cfg["hidden_size"])

## 5 — Tokenizer Qwen

Đếm token phải bằng đúng tokenizer của generator, không ước từ số từ: tiếng Việt luật
cho **1,314 token/từ** ở answer và 1,217 ở question (đo trên 6.908 cặp), nên "576 từ"
của p90 thật ra là 754 token. Sai số đó đủ để chọn nhầm `max_seq_len`.

Ưu tiên `tokenizer.json` của model đã gắn; không có thì tải Qwen2.5 (cùng dòng BPE 151k).


In [ ]:
from pathlib import Path

tk = next((p for p in Path("/kaggle/input").glob("**/tokenizer.json")
           if "qwen" in str(p).lower()), None)
if tk is None:
    from huggingface_hub import hf_hub_download
    tk = Path(hf_hub_download("Qwen/Qwen2.5-1.5B", "tokenizer.json"))
    print("KHÔNG thấy tokenizer trong input — tải Qwen2.5-1.5B thay thế:", tk)
os.environ["QWEN_TOKENIZER"] = str(tk)

from tokenizers import Tokenizer
t = Tokenizer.from_file(str(tk))
print("vocab", t.get_vocab_size(), "| kiểm:", len(t.encode("Căn cứ khoản 1 Điều 5 Nghị định 90/2017/NĐ-CP").ids), "token")


## 6 — Quét ngân sách

Năm cấu hình, **một** lần rank mỗi câu: rerank là phần đắt (50 cặp qua XLM-R large),
`chon_evidence` chỉ đọc lại list đã xếp hạng. Tính lại `rank()` cho từng cấu hình là
nhân thời gian GPU lên 5 lần mà không thêm con số nào.

Mốc BM25 thuần phải đấu (máy dev, n=300):

| top_n | max_chars | gold | p50 | >4096 |
|------:|----------:|-----:|----:|------:|
| 5 | 24.000 | 78,7% | 5.575 | 90,7% |
| 5 | 12.000 | 74,6% | 3.743 | 25,7% |
| 3 | 12.000 | 72,4% | 3.280 | 12,7% |
| 5 |  8.000 | 68,7% | 2.497 |  0,3% |

Cột `gold` phải TĂNG so với bảng này; không tăng nghĩa là reranker không giúp chọn
đúng Document, và mọi kết luận §15.3 phải xem lại.


In [ ]:
!python scripts/build_sft.py --do-token --n 300 --model $MODEL --batch-size 32 --pool 50
